In [1]:
# Capstone project work
# Author Padmanabhan S Pillai, Dec 1 2025
# Notebook works towards building the project, based on the course material
# Project would be deployed to Vertex AgentEngine
# This is not to a guide to building a production quality code but to work through concepts around buiding agents, in the current form.

# Modules required for Code development and deployment


In [2]:
import os
import random
import time
import vertexai
from kaggle_secrets import UserSecretsClient
from vertexai import agent_engines

print("✅ Development and Deployment module imports completed successfully")

✅ Development and Deployment module imports completed successfully


# Attach GCP account

using menu "Add-ons-->Google Cloud SDK" and completing the Oauth flow, Code below sets the temporary credentials to work with GCP services using GCP SDK

In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
user_credential = user_secrets.get_gcloud_credential()
user_secrets.set_tensorflow_credential(user_credential)

print("✅ GCP Cloud credentials configured")

✅ GCP Cloud credentials configured


# Set the GCP project

**Enable API for Vertex deployment**

It is important that for deploying successfully the agent to the Vertex AgentEngine, the following API & Services are enabled in the GCP project.
- Vertex AI API
- Cloud Storage API
- Cloud Logging API
- Cloud Monitoring API
- Cloud Trace API
- Telemetry API.
  
**Enable API for Map MCP service**

To work with maps MCP tools from google ,  https://mapstools.googleapis.com/mcp additional services listes below are enabled and the API key restrictioon list includes the below AP/Services**
- Maps Grounding Lite API
- Directions API
- Geocoding API
- Places API *( Please note not the Places API(New))*
- Routes API and
- Weather API

**Enable MCP service access for Maps Grounding Lite API**

Additionally it is important to make sure MCP service is enabled in the GCP project, for that execute the below command using gcloud client tool.**

```bash
gcloud beta services mcp enable mapstools.googleapis.com --project=<your-project-number>
```
*Please note that the project number is used in the above call, not the id*

# Set API key and GCP PROJECT_ID as OS environment variable

1. Using the menu "Add-ons--> Secrets" store the API key associated with the project in the kaggle secret manager and enable it by checking the box
2. Execute the following code to set the environment variable
3. Please note that code executing from Vertex AgentMachine does not require tke API key to work with LLM, at the same time the MCP service do need this.


In [4]:
import os
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
    GOOGLE_CLOUD_PROJECT = UserSecretsClient().get_secret("GOOGLE_CLOUD_PROJECT")
    os.environ["GOOGLE_CLOUD_PROJECT"] = GOOGLE_CLOUD_PROJECT
    print("✅ Setup environment variables with API Key and Project Id complete.")
except Exception as e:
    print(
        f"Error setting APIKEY and PROJECT ID from  Kaggle secrets to env variable. Details: {e}"
    )

✅ Setup environment variables with API Key and Project Id complete.


<img width="800" src="https://github.com/pnabhans/kaggle5dayAgentCaptstone/blob/5a975aa08d9389ccb19281fb418726b65c708530/img/AgentArchitecture.png?raw=true" alt="Concierge Agent" />

---

## Concierge Agent

Implemented using Multi Agent Architecture

## Expected skills
   Guest (aka user) interacts with the **Guest Manager** Agent
        - discuss about a new assistance required or updates about their decision about an earlier assistance requested awaiting a decision
   **Guest Manager** A new assistance is marked uniquely to address the specific assistance across session and a period of time
        - Based on the kind o
        

- **Model:** Uses gemini-2.5-flash-lite for low latency and cost-efficiency.
- **Tools:** Includes a `get_weather` function to demonstrate tool execution.
- **Persona:** Responds conversationally to prove the instruction-following capabilities.

This demonstrates the foundational ADK architecture we are about to package: **Agent + Tools + Instructions**.

We'll create the following files and directory structure:

```
sample_agent/
├── agent.py                  # The logic
├── requirements.txt          # The libraries
├── .env                      # The secrets/config
└── .agent_engine_config.json # The hardware specs
```